In [6]:
import os
os.getcwd()

from pathlib import Path
Path("build/pdd_dose.txt").exists()

False

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

def read_geant4_mesh_dump(path: Path) -> pd.DataFrame:
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if (not line) or line.startswith("#"):
                continue
            parts = [p.strip() for p in line.split(",")]
            if len(parts) < 6:
                continue
            ix = int(parts[0]); iy = int(parts[1]); iz = int(parts[2])
            val = float(parts[3]); val2 = float(parts[4]); entry = int(float(parts[5]))
            rows.append((ix, iy, iz, val, val2, entry))
    return pd.DataFrame(rows, columns=["iX","iY","iZ","val","val2","entry"])

In [5]:
# ---- USER: point to your PDD file ----
PDD_TXT = Path("build") / "pdd_100k_J50014575_T1.txt"  # change
# build/pdd_100k_J50014575_T1.txt
NBIN_Z = 160
HALF_Z_MM = 40.0
DEPTH_MODE = "center"  # center is correct for voxel centers

df = read_geant4_mesh_dump(PDD_TXT)

# PDD mesh should be 1×1×NBIN_Z, so sum over iX,iY by iZ
g = df.groupby("iZ", as_index=False)["val"].sum()
all_iz = pd.DataFrame({"iZ": np.arange(NBIN_Z)})
g = all_iz.merge(g, on="iZ", how="left").fillna(0.0)

dose = g["val"].to_numpy()
if dose.max() <= 0:
    raise RuntimeError("Dose is zero everywhere - check mesh placement or beamOn.")

# Depth axis
full_len = 2*HALF_Z_MM
dz = full_len / NBIN_Z
if DEPTH_MODE == "center":
    depth_mm = (np.arange(NBIN_Z) + 0.5) * dz
else:
    depth_mm = np.arange(NBIN_Z) * dz

pdd = dose / dose.max()

def r_at_percent(depth_mm, pdd, percent):
    imax = int(np.argmax(pdd))
    d2 = depth_mm[imax:]
    y2 = pdd[imax:]
    if np.all(y2 >= percent):
        return np.nan
    j = np.where(y2 < percent)[0][0]
    if j == 0:
        return d2[0]
    dA, dB = d2[j-1], d2[j]
    yA, yB = y2[j-1], y2[j]
    return dA + (percent - yA) * (dB - dA) / (yB - yA)

dmax_idx = int(np.argmax(pdd))
dmax_mm  = float(depth_mm[dmax_idx])
R90 = r_at_percent(depth_mm, pdd, 0.90)
R80 = r_at_percent(depth_mm, pdd, 0.80)
R50 = r_at_percent(depth_mm, pdd, 0.50)

plt.figure()
plt.plot(depth_mm, pdd, marker="o", linewidth=1)
plt.xlabel("Depth in water (mm)")
plt.ylabel("PDD (norm to dmax)")
plt.grid(True)
plt.ylim(0, 1.05)

plt.axvline(dmax_mm, linestyle="--")
plt.text(dmax_mm, 1.02, f"dmax={dmax_mm:.2f} mm", rotation=90, va="bottom")

for val, name in [(R90,"R90"), (R80,"R80"), (R50,"R50")]:
    if np.isfinite(val):
        plt.axvline(val, linestyle="--")
        plt.text(val, 0.02, f"{name}={val:.2f}", rotation=90, va="bottom")

plt.title(f"PDD: {PDD_TXT.name}")
plt.show()

print("dz =", dz)
print("dmax =", dmax_mm, "mm")
print("R90  =", R90, "mm")
print("R80  =", R80, "mm")
print("R50  =", R50, "mm")

FileNotFoundError: [Errno 2] No such file or directory: 'build/pdd_100k_J50014575_T1.txt'

In [ ]:
# ---- USER: point to your 2D file ----
EXIT2D_TXT = Path("build") / "exitFluence2D_N50000_A50014566_T1.txt"  # change
# 

NX, NY = 120, 120
HALF_X_MM, HALF_Y_MM = 60.0, 60.0   # match macro /score/mesh/boxSize

df = read_geant4_mesh_dump(EXIT2D_TXT)
df = df[df["iZ"] == 0].copy()

# fill missing voxels with 0
full = pd.MultiIndex.from_product([range(NX), range(NY)], names=["iX","iY"]).to_frame(index=False)
df = full.merge(df, on=["iX","iY"], how="left").fillna(0.0)

img = df.pivot(index="iY", columns="iX", values="val").to_numpy()

dx = 2*HALF_X_MM / NX
dy = 2*HALF_Y_MM / NY
x = (np.arange(NX) + 0.5)*dx - HALF_X_MM
y = (np.arange(NY) + 0.5)*dy - HALF_Y_MM

# Normalize for film-like appearance
m = img.max()
imgN = img/m if m > 0 else img

plt.figure()
extent = [x[0]-dx/2, x[-1]+dx/2, y[0]-dy/2, y[-1]+dy/2]
plt.imshow(imgN, origin="lower", extent=extent, aspect="equal")
plt.colorbar(label="normalized")
plt.xlabel("x (mm)")
plt.ylabel("y (mm)")
plt.title(f"Exit 2D map: {EXIT2D_TXT.name}")
plt.show()

In [ ]:
ix0 = NX//2
iy0 = NY//2

prof_x = img[iy0, :]
prof_y = img[:, ix0]

px = prof_x / prof_x.max() if prof_x.max() > 0 else prof_x
py = prof_y / prof_y.max() if prof_y.max() > 0 else prof_y

plt.figure()
plt.plot(x, px, label="X profile at y≈0")
plt.plot(y, py, label="Y profile at x≈0")
plt.xlabel("Position (mm)")
plt.ylabel("Normalized")
plt.grid(True)
plt.legend()
plt.title("Central profiles at exit plane")
plt.show()

In [ ]:
X, Y = np.meshgrid(x, y)
R = np.sqrt(X**2 + Y**2)

rmax = 60.0
dr = 0.5
rbins = np.arange(0, rmax+dr, dr)

rad = np.zeros(len(rbins)-1)
cnt = np.zeros(len(rbins)-1)

flatR = R.ravel()
flatV = img.ravel()

for i in range(len(rbins)-1):
    msk = (flatR >= rbins[i]) & (flatR < rbins[i+1])
    if np.any(msk):
        rad[i] = np.mean(flatV[msk])
        cnt[i] = np.sum(msk)

rcent = 0.5*(rbins[:-1]+rbins[1:])
radN = rad / np.max(rad) if np.max(rad) > 0 else rad

plt.figure()
plt.plot(rcent, radN)
plt.xlabel("r (mm)")
plt.ylabel("Radial mean (normalized)")
plt.grid(True)
plt.title("Radial mean at exit plane")
plt.show()